In [22]:
import requests
import pandas as pd
import numpy as np
import sqlalchemy
import pymysql
from keys import *

In [23]:
OWM_key = "b5e9be5d815066cf3145f32c702cfb7f"

city = "Mexico City"
country = "MX"

url = f'http://api.openweathermap.org/data/2.5/forecast/?q={city},{country}&appid={OWM_key}&units=metric&lang=en'
response = requests.get(url)
data = response.json()

In [24]:
forecast_list = data.get('list', [])

# Preparar listas
times = []
temperatures = []
humidities = []
weather_statuses = []
wind_speeds = []
rain_volumes = []
snow_volumes = []

for entry in forecast_list:
    times.append(entry.get('dt_txt', np.nan))
    temperatures.append(entry.get('main', {}).get('temp', np.nan))
    humidities.append(entry.get('main', {}).get('humidity', np.nan))
    weather_statuses.append(entry.get('weather', [{}])[0].get('main', np.nan))
    wind_speeds.append(entry.get('wind', {}).get('speed', np.nan))
    rain_volumes.append(entry.get('rain', {}).get('3h', np.nan))
    snow_volumes.append(entry.get('snow', {}).get('3h', np.nan))

In [25]:
df = pd.DataFrame({
    'weather_datetime': times,
    'temperature': temperatures,
    'humidity': humidities,
    'weather_status': weather_statuses,
    'wind': wind_speeds,
    'rain_qty': rain_volumes,
    'snow': snow_volumes,
})

# Añadir columna obligatoria para la tabla MySQL
df['municipality_iso_country'] = f"{city},{country}"

print(df.head())

      weather_datetime  temperature  humidity weather_status  wind  rain_qty  \
0  2025-11-03 12:00:00         8.13        78         Clouds  1.90       NaN   
1  2025-11-03 15:00:00        11.18        69         Clouds  2.08       NaN   
2  2025-11-03 18:00:00        16.91        31          Clear  3.12       NaN   
3  2025-11-03 21:00:00        21.40        20          Clear  4.02       NaN   
4  2025-11-04 00:00:00        18.89        27          Clear  5.69       NaN   

   snow municipality_iso_country  
0   NaN           Mexico City,MX  
1   NaN           Mexico City,MX  
2   NaN           Mexico City,MX  
3   NaN           Mexico City,MX  
4   NaN           Mexico City,MX  


In [26]:
schema = "gans"
host = "127.0.0.1"
user = "root"
password = "Misterio619"
port = 3306

con = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

# Probar conexión
try:
    engine = sqlalchemy.create_engine(con)
    with engine.connect() as conn:
        result = conn.execute("SELECT NOW();")
        for row in result:
            print("✅ Conexión exitosa, hora del servidor MySQL:", row[0])
except Exception as e:
    print("❌ Error de conexión:", e)


❌ Error de conexión: Not an executable object: 'SELECT NOW();'


In [27]:
try:
    df.to_sql('weather_data', if_exists='append', con=con, index=False)
    print("✅ Datos insertados correctamente en weather_data")
except Exception as e:
    print("❌ Error al insertar datos:", e)

✅ Datos insertados correctamente en weather_data
